# Generate sample signal

* This 'signal' is a dictionary of all alphaX-independent fundamental quantities

## Imports

In [ ]:
import photoproductionmodel as pm
import branching_ratios as br #be sure this script is using the correct stuff internally
import numpy as np
import scipy
import matplotlib.pyplot as plt

mp = 0.9382720813
e = 0.303
m_pi_charged = 0.140 #charged pion

## Photoproduction model params and gauge couplings

In [ ]:
gauge_couplings = {"A'":np.array([2/3,-1/3,-1/3,-1,-1,0,0,0]),
                   "B-L":np.array([1/3,1/3,1/3,-1,-1,-1,-1,-1]),
                   "B":np.array([1/3,1/3,1/3,-e**2/(4*np.pi)**2,-e**2/(4*np.pi)**2,0,0,0]),
                   "Chargephobic":np.array([-1/(3*np.sqrt(2)),np.sqrt(2)/3,np.sqrt(2)/3,0,0,-1/np.sqrt(2),-1/np.sqrt(2),-1/np.sqrt(2)])}
params = np.array([11.83140742,  4.33676463,  0.83162154,  0.77993622,  0.8288554 ,
        0.80163735,  0.63063899,  0.69853717,  2.52088083,  1.87247473,
        5.61516946,  3.12417096,  1.12279023,  0.99670773,  0.92126733,
        0.97984822,  0.85205764,  1.06938238,  2.1765273 ,  3.38643496,
        0.78531502,  3.64469317,  0.9668222 ])

## Sampling kinematic variable methods

In [ ]:
def P(s_tot,EX,cosX,params,mX,xq):
  if cosX > 1:
    return 0
  return pm.dsigX_dEX_dcosX(s_tot,EX,cosX,params,mX,xq)

def mcmc_sample_EX_cosX(n,s_tot,params,mX,xq):
  '''
  return
    ndarray : nx2 posterior distribution (EX [GeV], cosX)
  n
    int : number of samples
  s_tot
    float : i genuinely do not fucking know
  params
    ndarray floats : photoproduction model parameters
  mX
    float : X boson mass
  xq
    ndarray floats : quark couplings
  '''

  x_samples = np.zeros((n, 2))

  E_beam = (s_tot - mp**2) / (2 * mp)

  central_E = (E_beam - mX) / 2

  standard_deviation_E = (E_beam-central_E)*0.67
  standard_deviation_cos = 0.01

  # starting point
  x_samples[0,0] = central_E
  x_samples[0,1] = 0.99

  counter = 0
  for i in range(n - 1):
    EX_old = x_samples[i,0]
    cosX_old = x_samples[i,1]

    ## proposal ##
    EX_new = np.random.normal(EX_old,standard_deviation_E)
    cosX_new = np.random.normal(cosX_old,standard_deviation_cos)

    ## accept/reject ##

    A = P(s_tot,EX_new,cosX_new,params,mX,xq) / P(s_tot,EX_old,cosX_old,params,mX,xq)

    if A > 1:
      x_samples[i+1,0] = EX_new
      x_samples[i+1,1] = cosX_new
      counter +=1

    elif A > np.random.uniform(0,1):
      x_samples[i+1,0] = EX_new
      x_samples[i+1,1] = cosX_new
      counter +=1

    else:
      x_samples[i+1,0] = EX_old
      x_samples[i+1,1] = cosX_old

  return x_samples

In [ ]:
def sample_pion_four_momenta_in_X(n,mX):
  '''
  return
     P1,P2 : tuple of nd arrays
      corresponding pion four momenta arrays in rest frame of X boson (P2 = -P1)
  n
    int | number of samples
  mX
    float | X boson mass
  '''
  E_pi = mX / 2
  pi_momentum_norm = np.sqrt(E_pi**2-m_pi_charged**2)
  pi_momentum = np.zeros((n,4))

  for i in range(n):
    cos_alpha = np.random.uniform(-1,1)
    sin_alpha = np.sqrt(1 - cos_alpha**2) # ignoring that sin could be pos or neg

    phi = np.random.uniform(0,2*np.pi)

    pi_momentum[i,0] = E_pi #natural units
    pi_momentum[i,1] = sin_alpha * np.cos(phi) * pi_momentum_norm
    pi_momentum[i,2] = sin_alpha * np.sin(phi) * pi_momentum_norm
    pi_momentum[i,3] = cos_alpha * pi_momentum_norm

  return [pi_momentum, pi_momentum*[1,-1,-1,-1]]

def transform_restX_to_lab(EX_samples, cosX_samples, mX, pion_four_momenta_samples_in_X):
  '''
  return
    P1,P2 : tuple of nd arrays
      pi+ , pi- four momenta in lab
  '''

  pi_plus_four_momenta_samples_in_X , pi_minus_four_momenta_samples_in_X = pion_four_momenta_samples_in_X

  assert len(EX_samples) == len(pi_plus_four_momenta_samples_in_X) , "cosX, EX sample size doesnt match pion momenta sample size"

  N = len(EX_samples)

  pi_plus_four_momenta_samples_in_lab = np.zeros((N,4))
  pi_minus_four_momenta_samples_in_lab = np.zeros((N,4))

  ## TRANSFORM MATRICES ##
  for i in range(N):
    gamma = EX_samples[i] / mX
    beta = np.sqrt(1 - 1/gamma**2)

    cX = cosX_samples[i]
    sX = np.sqrt(1 - cX**2)

    Lambda = np.array([
        [gamma, 0,  0, gamma*beta],
        [0,   1,  0,  0],
        [0, 0,  1, 0],
        [beta*gamma, 0,  0, gamma]
    ])
    R = np.array([
        [1, 0,  0, 0],
        [0, cX, 0,  -sX],
        [0, 0,  1, 0],
        [0, sX, 0, cX]
    ])

    pi_plus_four_momenta_samples_in_lab[i] = R@Lambda@pi_plus_four_momenta_samples_in_X[i]
    pi_minus_four_momenta_samples_in_lab[i] = R@Lambda@pi_minus_four_momenta_samples_in_X[i]

  return pi_plus_four_momenta_samples_in_lab, pi_minus_four_momenta_samples_in_lab

## Generate dataset of kinematic samplesm branching ratios, lifetimes, and differential crossections

In [ ]:
#mass
mX = 0.7

#beam energy
s_tot = 42

#gauge params (defined above)

#photoproduction params (defined above)

#number of samples
n = 100

#EX and cosX samples
EX_cosX_samples = mcmc_sample_EX_cosX(n,s_tot,params,mX,gauge_couplings['Chargephobic'][0:3])
EX_samples = EX_cosX_samples[:,0]
costhX_samples = EX_cosX_samples[:,1]

#pi+pi- four momentum in x boson frame samples
pi_plus_P4_samples_in_X_frame , pi_minus_P4_samples_in_X_frame = sample_pion_four_momenta_in_X(n,mX)

#integrated cross section
sigma_X = 10

#branching ratio pi+pi-
BR = br.decay_profile(mX,1,gauge_couplings['Chargephobic'])['BR_pi_plus_pi_minus']

#dictionary
data = {'EX_samples':EX_samples,
        'costhX_samples':costhX_samples,
        'pi_plus_P4_samples_in_X_frame':pi_plus_P4_samples_in_X_frame,
        'pi_minus_P4_samples_in_X_frame':pi_minus_P4_samples_in_X_frame,
        'mX':mX,
        'sigma_X/alphaX':sigma_X,
        'BR_pi_plus_pi_minus':BR,
        'gauge_couplings':gauge_couplings,
        'photo_params':params,
        's_tot':s_tot
          }

np.savez(f'sampling_data_mX_0p{str(mX)[-1]}.npz',**data)

## Load mX dataset

In [ ]:
load = dict(np.load('sampling_data_mX_0p7.npz',allow_pickle=True))

# Calculate expected number of events for new U(1) gauge boson in fixed proton target electron beam collider experiment as a function of gauge parameters and boson mass

## Class

In [ ]:
class signal_data:

  import branching_ratios as br

  def __init__(self,filename):

    data = dict(np.load(filename,allow_pickle=True))
    for key,values in data.items():
      setattr(self,key,values)     # assert presence of required data

    self.transform()
    self.set_alphaX_independent_decay_length()

  def transform(self):

    ## Transforming X frame pion momentum to lab frame (wat modulars)
    N = len(EX_samples)
    pi_plus_P4_samples_in_LAB_frame = np.zeros((N,4))
    pi_minus_P4_samples_in_LAB_frame = np.zeros((N,4))
    for i in range(N):
      gamma = self.EX_samples[i] / mX
      beta = np.sqrt(1 - 1/gamma**2)

      cX = self.costhX_samples[i]
      sX = np.sqrt(1 - cX**2)

      Lambda = np.array([
          [gamma, 0,  0, gamma*beta],
          [0,   1,  0,  0],
          [0, 0,  1, 0],
          [beta*gamma, 0,  0, gamma]
      ])
      R = np.array([
          [1, 0,  0, 0],
          [0, cX, 0,  -sX],
          [0, 0,  1, 0],
          [0, sX, 0, cX]
      ])

      pi_plus_P4_samples_in_LAB_frame[i] = R@Lambda@self.pi_plus_P4_samples_in_X_frame[i]
      pi_minus_P4_samples_in_LAB_frame[i] = R@Lambda@self.pi_minus_P4_samples_in_X_frame[i]

    setattr(self,'pi_plus_P4_samples_in_LAB_frame',pi_plus_P4_samples_in_LAB_frame)
    setattr(self,'pi_minus_P4_samples_in_LAB_frame',pi_minus_P4_samples_in_LAB_frame)

  def set_alphaX_independent_decay_length(self):
    setattr(self,
            'c_tau_divby_alphaX',
            br.get_decay_length(self.mX.item(),1,self.gauge_couplings.item()['Chargephobic']))

  def apply_resolution(self,sigma_E,sigma_p):

    p_plus = self.pi_plus_P4_samples_in_LAB_frame[:,1:]
    p_minus = self.pi_minus_P4_samples_in_LAB_frame[:,1:]
    E_plus = self.pi_plus_P4_samples_in_LAB_frame[:,0]
    E_minus = self.pi_minus_P4_samples_in_LAB_frame[:,0]
    p_plus_x = p_plus[:,0]
    p_plus_y = p_plus[:,1]
    p_plus_z = p_plus[:,2]
    p_minus_x = p_minus[:,0]
    p_minus_y = p_minus[:,1]
    p_minus_z = p_minus[:,2]

    mX_sq_reconstructed_arr = np.zeros(len(p_plus))
    for i in range(len(p_plus)):
      E_plus_measured = np.random.normal(E_plus[i],sigma_E)
      E_minus_measured = np.random.normal(E_minus[i], sigma_E)
      p_plus_x_measured = np.random.normal(p_plus_x[i], sigma_p)
      p_plus_y_measured = np.random.normal(p_plus_y[i], sigma_p)
      p_plus_z_measured = np.random.normal(p_plus_z[i], sigma_p)
      p_minus_x_measured = np.random.normal(p_minus_x[i], sigma_p)
      p_minus_y_measured = np.random.normal(p_minus_y[i], sigma_p)
      p_minus_z_measured = np.random.normal(p_minus_z[i], sigma_p)
      p_plus_measured = np.array([p_plus_x_measured,p_plus_y_measured,p_plus_z_measured])
      p_minus_measured = np.array([p_minus_x_measured,p_minus_y_measured,p_minus_z_measured])
      mX_sq_reconstructed_arr[i] = (E_plus_measured+E_minus_measured)**2 - np.linalg.norm(p_plus_measured + p_minus_measured)**2

    quant_0p05,quant_0p95 = np.quantile(mX_sq_reconstructed_arr,[0.05,0.95])
    mask = (mX_sq_reconstructed_arr > quant_0p95) | (mX_sq_reconstructed_arr < quant_0p05)
    mX_sq_reconstructed_arr[mask] = 0
    acceptance = np.count_nonzero(mX_sq_reconstructed_arr) / len(mX_sq_reconstructed_arr)

    setattr(self,'5quantile',quant_0p05)
    setattr(self,'95quantile',quant_0p95)
    setattr(self,'mX_sq_reconstructed_arr',mX_sq_reconstructed_arr)
    setattr(self,'acceptance_mX_sq',acceptance)
    print(f'acceptance: {acceptance}')
    return

  def A(self,cosX,EX,L0,mX,x,c_tau,l0=None):
    '''
    return | displaced production acceptance-fraction
    '''
    print(f'cosX:{cosX},EX:{EX}')
    if cosX < 0:
      return np.nan
    gamma = EX/mX
    if not l0:
      return np.exp(-L0/(gamma*c_tau*cosX))
    elif l0:
      print('have not tested the l0 implementation')
      return np.exp(-L0/(gamma*c_tau*cosX)) - np.exp(-l0/(gamma*c_tau*cosX))

  def estimate_expected_A(self,alphaX,L0,l0=None):
    '''
    return | cross section weighted average displaced production acceptance fraction
    L0,l0 | float : distance from production point to end of shield (L0) and start of shield (l0)
    c_tau | float : proper decay length
    x | arr of floats : gauge couplings [u,d,s,e,mu,nu_e,nu_mu,nu_tau]
    '''

    EX_samples = self.EX_samples
    cosX_samples = self.costhX_samples

    c_tau = self.c_tau_divby_alphaX * alphaX / 1e13 #fm to cm

    A_samples = np.zeros(len(EX_samples))
    for i in range(len(EX_samples)):
      A_samples[i] = A(cosX_samples[i],
                       EX_samples[i],
                       L0,
                       self.mX,
                       self.gauge_couplings.item()['Chargephobic'],
                       c_tau
                       ) #returning zero

    A_samples[np.isnan(A_samples)] = 0 # negative cosX values are returned as NaN by A().

    # print(A_samples)
    return np.sum(A_samples) / len(A_samples)

  def plt_num_events(self,alphaX):


    return

s = signal_data('sampling_data_mX_0p7.npz')
s.apply_resolution(0.01,0.01)
vars(s).keys()
s.estimate_expected_A(1e-8,10)

acceptance: 0.9


np.float64(0.48931492986914604)

## Imports

In [ ]:
import photoproductionmodel as pm
import branching_ratios as br #be sure this script is using the correct stuff internally
import numpy as np
import scipy
import matplotlib.pyplot as plt

mp = 0.9382720813
e = 0.303
m_pi_charged = 0.140 #charged pion

## Photoproduction model params and gauge couplings

In [ ]:
gauge_couplings = {"A'":np.array([2/3,-1/3,-1/3,-1,-1,0,0,0]),
                   "B-L":np.array([1/3,1/3,1/3,-1,-1,-1,-1,-1]),
                   "B":np.array([1/3,1/3,1/3,-e**2/(4*np.pi)**2,-e**2/(4*np.pi)**2,0,0,0]),
                   "Chargephobic":np.array([-1/(3*np.sqrt(2)),np.sqrt(2)/3,np.sqrt(2)/3,0,0,-1/np.sqrt(2),-1/np.sqrt(2),-1/np.sqrt(2)])}
params = np.array([11.83140742,  4.33676463,  0.83162154,  0.77993622,  0.8288554 ,
        0.80163735,  0.63063899,  0.69853717,  2.52088083,  1.87247473,
        5.61516946,  3.12417096,  1.12279023,  0.99670773,  0.92126733,
        0.97984822,  0.85205764,  1.06938238,  2.1765273 ,  3.38643496,
        0.78531502,  3.64469317,  0.9668222 ])

## Decay Length Acceptance

In [ ]:
def A(cosX,EX,L0,mX,x,c_tau,l0=None):
  '''
  returns
    A(cosX,EX) the acceptance fraction or probability of decaying after shield and (if specified) before detector
  L0
    float distance from production point to end of shield
  c_tau
    float proper decay length
  mX
    float mass of X boson
  x
    ndarray of floats gauge couplings [u,d,s,e,mu,nu_e,nu_mu,nu_tau]
  '''

  if cosX < 0:
    return np.nan

  gamma = EX/mX

  if not l0:
    return np.exp(-L0/(gamma*c_tau*cosX))

  elif l0:
    print('nonzero l0 is not tested yet')
    return np.exp(-L0/(gamma*c_tau*cosX)) - np.exp(-l0/(gamma*c_tau*cosX))

## Integrating differential cross section / gauge coupling strength

In [ ]:
def dsig_integrand(EX,cosX,s_tot,params,mX,xq):
  '''
  integrand formatted for dblquad integrating
  '''
  return pm.dsigX_dEX_dcosX(s_tot,EX,cosX,params,mX,xq)

In [ ]:
def integrate_dsig(s_tot,params,mX,xq):
  '''
  return
    float : mu_barns integrated cross section
  '''
  cosX_lower_boundary = 0
  cosX_upper_boundary = 1
  EX_lower_boundary_func = mX + 1e-6
  EX_upper_boundary_func = (s_tot - mp**2) / (2 * mp)

  result , error = scipy.integrate.dblquad(
    dsig_integrand,
    cosX_lower_boundary,
    cosX_upper_boundary,
    EX_lower_boundary_func,
    EX_upper_boundary_func,
    args=(s_tot,params,mX,xq)
    )

  return result #true value has a factor of alphaX here

# Notes

* Include parameters, gauge couplings, and beam energy in the data file.
* Your choice on whether or not to include the X frame pion four momenta samples
* Save BR and decay length seperately
* After implementing resolution, decide window of invariant mass and count for fraction
* can decide window with quantiles